In [ ]:
import os
import sys
import re

def check_train_folder(train_folder):
    # Get all scenario folders in the train folder.
    scenarios = [d for d in os.listdir(train_folder) if os.path.isdir(os.path.join(train_folder, d))]
    required_directions = {'0', '1', '2', '3', '4', '5'}

    # Overall report for missing radar files.
    overall_missing_data = {}  # { scenario: { agent: { timestamp: [missing_directions] } } }
    # Overall agent timestamps per scenario.
    overall_agent_timestamps = {}  # { scenario: { agent: [timestamp, ...] } }

    # Regex to match files: <timestamp>_radar_<direction>.pcd
    pattern = re.compile(r'^(?P<timestamp>\d+)_radar(?P<direction>0|1|2|3|4|5)\.npy$')

    for scenario in scenarios:
        scenario_path = os.path.join(train_folder, scenario)
        agents = [d for d in os.listdir(scenario_path) if os.path.isdir(os.path.join(scenario_path, d))]
        scenario_missing = {}          # missing radar info for the scenario
        agent_timestamp_dict = {}      # {agent: {timestamp: set(directions)} }
        scenario_agent_timestamps = {} # {agent: set(timestamps)}
        global_timestamps = set()      # union of all timestamps in this scenario

        # Process each agent in the scenario.
        for agent in agents:
            agent_path = os.path.join(scenario_path, agent)
            agent_timestamp_dict[agent] = {}
            scenario_agent_timestamps[agent] = set()
            for file in os.listdir(agent_path):
                match = pattern.match(file)
                if match:
                    timestamp = match.group("timestamp")
                    direction = match.group("direction")
                    scenario_agent_timestamps[agent].add(timestamp)
                    if timestamp not in agent_timestamp_dict[agent]:
                        agent_timestamp_dict[agent][timestamp] = set()
                    agent_timestamp_dict[agent][timestamp].add(direction)
                    global_timestamps.add(timestamp)

        # Save each agent's found timestamps for reporting.
        overall_agent_timestamps[scenario] = {agent: sorted(list(ts_set)) for agent, ts_set in scenario_agent_timestamps.items()}

        # For every agent, check every timestamp (from the union) for missing radar files.
        for agent, ts_dict in agent_timestamp_dict.items():
            for ts in global_timestamps:
                if ts in ts_dict:
                    missing_dirs = required_directions - ts_dict[ts]
                    if missing_dirs:
                        if agent not in scenario_missing:
                            scenario_missing[agent] = {}
                        scenario_missing[agent][ts] = sorted(missing_dirs)
                else:
                    # Agent does not have any file for this timestamp.
                    if agent not in scenario_missing:
                        scenario_missing[agent] = {}
                    scenario_missing[agent][ts] = sorted(required_directions)

        # Detailed report for the scenario.
        print(f"Scenario: {scenario}")
        for agent in agents:
            print(f"  Agent: {agent}")
            timestamps_found = sorted(list(scenario_agent_timestamps.get(agent, set())))
            print(f"    Timestamps found: {', '.join(timestamps_found) if timestamps_found else 'None'}")
            if agent in scenario_missing:
                for ts, missing_dirs in sorted(scenario_missing[agent].items()):
                    print(f"    Timestamp {ts}: missing {', '.join(missing_dirs)}")
            else:
                print("    All timestamps have complete radar files.")

        # Check for consistency of timestamps among agents.
        unique_timestamp_sets = set(frozenset(ts) for ts in scenario_agent_timestamps.values())
        if len(unique_timestamp_sets) == 1:
            print("    All agents have the same timestamps.")
        else:
            print("    Inconsistent timestamps among agents:")
            for agent, ts_set in scenario_agent_timestamps.items():
                print(f"      {agent}: {sorted(ts_set)}")
        print("-" * 50)

        if scenario_missing:
            overall_missing_data[scenario] = scenario_missing

    # Summary report of missing radar files.
    if overall_missing_data:
        print("\nSummary of Missing Radar Files:")
        for scenario, agents_data in overall_missing_data.items():
            for agent, timestamps in agents_data.items():
                for ts, missing_dirs in sorted(timestamps.items()):
                    print(f"Scenario: {scenario}, Agent: {agent}, Timestamp: {ts} -> missing {', '.join(missing_dirs)}")
    else:
        print("All scenarios and agents have complete radar files for each timestamp.")



In [ ]:
 check_train_folder("/home/ws-ids-es3-01/Developer/Dataset/Dataset_OPV2V/train_additional")

In [ ]:
import os

def collect_timestamps(directory):
    """
    Walk through a directory structure and collect timestamps for each agent.
    :param directory: The root directory to walk.
    :return: Dictionary of {scenario: {agent: set of timestamps}}.
    """
    timestamps = {}
    for scenario in os.listdir(directory):
        scenario_path = os.path.join(directory, scenario)
        if not os.path.isdir(scenario_path):
            continue

        timestamps[scenario] = {}
        for agent in os.listdir(scenario_path):
            agent_path = os.path.join(scenario_path, agent)
            if not os.path.isdir(agent_path):
                continue

            timestamps[scenario][agent] = set()
            for file in os.listdir(agent_path):
                if file.endswith(".npy"):  # Adjust extension if needed
                    # Extract timestamp from filename (assumes <timestamp>_ prefix)
                    timestamp = file.split("_")[0]
                    timestamps[scenario][agent].add(timestamp)
    return timestamps


def compare_timestamps(dir1, dir2):
    """
    Compare timestamps between two directories.
    :param dir1: First directory to compare.
    :param dir2: Second directory to compare.
    """
    timestamps1 = collect_timestamps(dir1)
    timestamps2 = collect_timestamps(dir2)

    all_scenarios = set(timestamps1.keys()) | set(timestamps2.keys())

    for scenario in sorted(all_scenarios):
        agents_dir1 = timestamps1.get(scenario, {})
        agents_dir2 = timestamps2.get(scenario, {})
        all_agents = set(agents_dir1.keys()) | set(agents_dir2.keys())

        print(f"Scenario: {scenario}")
        for agent in sorted(all_agents):
            ts_dir1 = agents_dir1.get(agent, set())
            ts_dir2 = agents_dir2.get(agent, set())

            if ts_dir1 != ts_dir2:
                extra_in_dir1 = ts_dir1 - ts_dir2
                extra_in_dir2 = ts_dir2 - ts_dir1
                print(f"  Agent: {agent}")
                if extra_in_dir1:
                    print(f"    Extra timestamps in {dir1}: {sorted(extra_in_dir1)}")
                if extra_in_dir2:
                    print(f"    Extra timestamps in {dir2}: {sorted(extra_in_dir2)}")
            else:
                print(f"  Agent: {agent} -> Timestamps are consistent.")
        print("-" * 50)

In [ ]:
# Paths to compare
dir1 = "/home/ws-ids-es3-01/Developer/Dataset/Dataset_OPV2V/train"
dir2 = "/home/ws-ids-es3-01/Developer/Dataset/Dataset_OPV2V/train_additional"

compare_timestamps(dir1, dir2)

In [1]:
import os

def count_files_and_compare(dir1, dir2):
    def count_agent_files(directory, divisor):
        """
        Count files per agent folder in each scenario and divide by the given divisor.
        :param directory: Directory to process.
        :param divisor: The divisor to use for normalization.
        :return: Dictionary of {scenario: {agent: file_count // divisor}}.
        """
        agent_file_counts = {}
        for scenario in os.listdir(directory):
            scenario_path = os.path.join(directory, scenario)
            if not os.path.isdir(scenario_path):
                continue

            agent_file_counts[scenario] = {}
            for agent in os.listdir(scenario_path):
                agent_path = os.path.join(scenario_path, agent)
                if os.path.isdir(agent_path):
                    file_count = len([f for f in os.listdir(agent_path) if os.path.isfile(os.path.join(agent_path, f))])
                    agent_file_counts[scenario][agent] = file_count // divisor
        return agent_file_counts

    def compare_counts(counts1, counts2):
        """
        Compare file counts between two directories.
        :param counts1: File counts for dir1.
        :param counts2: File counts for dir2.
        """
        all_scenarios = set(counts1.keys()) | set(counts2.keys())

        for scenario in sorted(all_scenarios):
            agents1 = counts1.get(scenario, {})
            agents2 = counts2.get(scenario, {})
            all_agents = set(agents1.keys()) | set(agents2.keys())

            print(f"Scenario: {scenario}")
            for agent in sorted(all_agents):
                count1 = agents1.get(agent, 0)
                count2 = agents2.get(agent, 0)

                print(f"  Agent: {agent}")
                print(f"    Files in {dir1} (divided by 6): {count1}")
                print(f"    Files in {dir2} (divided by 7): {count2}")
                if count1 != count2:
                    print(f"    ** Difference: {count1 - count2} **")
                else:
                    print(f"    Counts are consistent.")
            print("-" * 50)

    # Count files in both directories
    counts_dir1 = count_agent_files(dir1, 6)
    counts_dir2 = count_agent_files(dir2, 7)

    # Compare the counts
    compare_counts(counts_dir1, counts_dir2)

In [4]:
# Paths to the directories
dir1 = "/home/ws-ids-es3-01/Developer/Dataset/Dataset_OPV2V/validate"
dir2 = "/home/ws-ids-es3-01/Developer/Dataset/Dataset_OPV2V/validate_additional"

count_files_and_compare(dir1, dir2)

Scenario: 2021_08_20_21_48_35
  Agent: 2149
    Files in /home/ws-ids-es3-01/Developer/Dataset/Dataset_OPV2V/validate (divided by 6): 112
    Files in /home/ws-ids-es3-01/Developer/Dataset/Dataset_OPV2V/validate_additional (divided by 7): 112
    Counts are consistent.
  Agent: 2158
    Files in /home/ws-ids-es3-01/Developer/Dataset/Dataset_OPV2V/validate (divided by 6): 112
    Files in /home/ws-ids-es3-01/Developer/Dataset/Dataset_OPV2V/validate_additional (divided by 7): 112
    Counts are consistent.
  Agent: 2167
    Files in /home/ws-ids-es3-01/Developer/Dataset/Dataset_OPV2V/validate (divided by 6): 112
    Files in /home/ws-ids-es3-01/Developer/Dataset/Dataset_OPV2V/validate_additional (divided by 7): 112
    Counts are consistent.
  Agent: 2176
    Files in /home/ws-ids-es3-01/Developer/Dataset/Dataset_OPV2V/validate (divided by 6): 112
    Files in /home/ws-ids-es3-01/Developer/Dataset/Dataset_OPV2V/validate_additional (divided by 7): 112
    Counts are consistent.
  Agent: 2